In [ ]:
import sys
sys.path.append('../')
from op_utility import reindex_structure
# Example usage of reindex_structure, which is to avoid RMSD calculation errors due to non-sequential residue numbering
source_file = 'input.pdb'
copied_file = 'output.pdb'
reindex_structure(source_file, copied_file)

## Enzyme Design Workflow

Enzyme design can be separated into two phases:

---

### 1. From-Scratch Backbone Design

In the first phase, you generate protein backbone structures *de novo*.

After designing the backbone, you should perform filtering to remove poor designs. Key criteria include:

- **Ligand SASA (Solvent Accessible Surface Area)**  
  Ensure the ligand is appropriately buried or exposed depending on your design goal.

- **Distance between three histidines and the ligand**  
  Check whether the catalytic residues (e.g., His residues) are positioned correctly relative to the ligand.

You can use the following code (provided separately) to perform this filtering step.

---

### 2. Post-Processing and Optimization

After filtering:

- **Rename your designs**  
  This is important to remove the suffix automatically added by HalluDesign.  
  Renaming ensures compatibility with downstream steps.

- **Run a second round of HalluDesign optimization**  
  Use the filtered and renamed structures as input for further refinement.

---

### Notes

- Filtering is critical to reduce computational cost in later stages.
- Proper residue–ligand geometry is essential for functional enzyme design.

In [ ]:
import pandas as pd
import numpy as np
from Bio.PDB import MMCIFParser
from tqdm import tqdm
import freesasa
import tempfile
import os

# ------------------ distance ------------------
def calc_min_dist(structure):
    zn_coords = []
    o_coords = []

    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    
                    # ZN
                    if atom.element == "ZN" or atom.get_name().startswith("ZN"):
                        zn_coords.append(atom.get_coord())

                    # C chain O1/O5
                    if chain.id == "C" and atom.get_name() in ["O2", "O5"]:
                        o_coords.append(atom.get_coord())

    if len(zn_coords) == 0 or len(o_coords) == 0:
        return None

    zn_coords = np.array(zn_coords)
    o_coords = np.array(o_coords)

    dists = np.linalg.norm(
        zn_coords[:, None, :] - o_coords[None, :, :],
        axis=-1
    )

    return np.min(dists)


# ------------------ SASA ------------------
from Bio.PDB.SASA import ShrakeRupley

def calc_chainC_sasa_biopython(structure):

    sr = ShrakeRupley()

    sr.compute(structure, level="A")

    sasa = 0.0

    for model in structure:
        for chain in model:
            if chain.id != "C":
                continue
            for residue in chain:
                for atom in residue:
                    if hasattr(atom, "sasa"):
                        sasa += atom.sasa

    return sasa

def count_his_near_zn(structure, cutoff=3.0):

    zn_coords = []
    his_residues = set()  

    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    # 找 ZN
                    if atom.element == "ZN" or atom.get_name().startswith("ZN"):
                        zn_coords.append(atom.get_coord())

    if len(zn_coords) == 0:
        return 0

    zn_coords = np.array(zn_coords)


    for model in structure:
        for chain in model:
            if chain.id != "A":
                continue

            for residue in chain:
                if residue.get_resname() != "HIS":
                    continue


                for atom in residue:
                    if atom.get_name() not in ["ND1", "NE2"]:
                        continue

                    coord = atom.get_coord()

                    dists = np.linalg.norm(zn_coords - coord, axis=1)

                    if np.any(dists <= cutoff):
                        his_residues.add((chain.id, residue.id[1]))
                        break  

    return len(his_residues)

# ------------------ Main Function ------------------
def process_csv(csv_path, out_csv):
    parser = MMCIFParser(QUIET=True)
    df = csv_path

    min_dists = []
    sasas = []
    his_counts = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing CIF"):
        cif_path = row["eval_path"]

        try:
            structure = parser.get_structure("struct", cif_path)

            dist = calc_min_dist(structure)
            sasa = calc_chainC_sasa_biopython(structure)
            his_n = count_his_near_zn(structure)

        except Exception as e:
            print(f"Error: {cif_path}, {e}")
            dist, sasa = None, None

        min_dists.append(dist)
        sasas.append(sasa)
        his_counts.append(his_n)

    df["min_ZN_O_dist"] = min_dists
    df["chainC_SASA"] = sasas
    df["ZN_HIS_count_3A"] = his_counts

    df.to_csv(out_csv, index=False)
    print(f"Saved to {out_csv}")

df1 = pd.read_csv("...../op_1/processing_results.csv")
df2 = pd.read_csv("...../op_2/processing_results.csv")
df = pd.concat([df1,df2])

df_f = df[(df["eval_A_plddt"] >= 85) & (df["eval_B_plddt"] >= 80)& (df["eval_iptm"] >= 0.80) ] # &  (df["eval_C_plddt"] >= 80) 
print(len(df))
print(len(df_f))
process_csv(df_f,"...../redesign_score.csv")




# Binding Affinity Optimization Proposal
1 Copy the desired design and generate 2,000 copies by running the following Python script.


2 Run the next column to obtain randomly fixed biases.

In [3]:
import os, shutil

src = "ligand_binder/protein_ligand.pdb"          
dst_dir = "ligand_binder/copys"      
os.makedirs(dst_dir, exist_ok=True)

name, ext = os.path.splitext(os.path.basename(src))

for i in range(1, 2001):
    shutil.copy(src, f"{dst_dir}/{name}_{i}{ext}")


In [4]:
import os
import random
import json
import pandas as pd
from Bio import PDB
from Bio.Data import IUPACData
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# ================= Parameter Configuration =================
# Bias range (only applies positive enhancement to real amino acids)
BIAS_MIN, BIAS_MAX = 1.0, 2.0

# Number of worker threads
NUM_THREADS = 8

# Framework regions (these residues will be skipped)
# frameworks = [
#     "QVKLEESGGGSVQTGGSLRLTCAAS",
#     "MGWFRQAPGKEREFVSGISWRGDSTGYADSVKGRFTIS",
#     "TVDLQMNSLKPEDTAIYYCAAAAG",
#     "YWGQGTQVTVSS"
# ]

# frameworks = [
#     "QVQLQESGGGLVQPGGSLRLSCAA",
#     "YAMGWFRQAPGKQREFVAAIRWSGGYTYYTDSVKGRFTISR",
#     "VYLQMNSLKPEDTAVYYCAATY",
#     "DYWGQGTQVTVSS"
# ]

frameworks = [
]

# Mapping from 3-letter amino acid codes to 1-letter codes
aa_3to1 = {k.upper(): v for k, v in IUPACData.protein_letters_3to1.items()}


# ================= Helper Functions =================
def is_in_framework(seq, res_idx, frameworks):
    """Check whether a residue is located within framework regions."""
    for f in frameworks:
        idx = seq.find(f)
        if idx != -1 and idx <= res_idx < idx + len(f):
            return True
    return False


def generate_bias_for_chain(chain):
    """Generate random bias values for chain A residues (only enhances native amino acids)."""
    seq_1letter = ""

    # Convert chain sequence into 1-letter amino acid representation
    for res in chain:
        if not PDB.is_aa(res):
            continue
        resname = res.resname.upper()
        seq_1letter += aa_3to1.get(resname, "X")

    bias_dict = {}

    for i, res in enumerate(chain):
        if not PDB.is_aa(res):
            continue

        res_id = res.id[1]
        res_key = f"A{res_id}"

        # Skip residues in framework regions
        if is_in_framework(seq_1letter, i, frameworks):
            continue

        resname = res.resname.upper()
        one = aa_3to1.get(resname, None)

        if one is None or one == "X":
            continue

        # Assign random bias only to the native amino acid at this position
        bias_strength = round(random.uniform(BIAS_MIN, BIAS_MAX), 2)
        bias_dict[res_key] = {one: bias_strength}

    return bias_dict


def process_single_pdb(pdb_path):
    """Process a single PDB file and return the bias dictionary."""
    parser = PDB.PDBParser(QUIET=True)
    pdb_file = os.path.basename(pdb_path)

    try:
        structure = parser.get_structure(pdb_file, pdb_path)
    except Exception as e:
        return {"file_path": pdb_path, "bias": None, "error": str(e)}

    # Extract chain A
    chain_A = None
    for model in structure:
        for chain in model:
            if chain.id == "A":
                chain_A = chain
                break

    if chain_A is None:
        return {"file_path": pdb_path, "bias": None, "error": "no_chain_A"}

    bias = generate_bias_for_chain(chain_A)

    return {
        "file_path": pdb_path,
        "bias": json.dumps(bias, ensure_ascii=False, separators=(",", ":")),
        "error": None
    }


def process_pdb_folder(folder_path, output_csv="bias_summary.csv"):
    """Multi-threaded PDB folder processing with progress bar."""
    pdb_files = [
        os.path.join(folder_path, f)
        for f in os.listdir(folder_path)
        if f.endswith(".pdb")
    ]

    if not pdb_files:
        print("❌ No PDB files found in the folder.")
        return

    results = []

    # Parallel processing using thread pool
    with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
        futures = {
            executor.submit(process_single_pdb, pdb_path): pdb_path
            for pdb_path in pdb_files
        }

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Processing PDBs",
            ncols=100
        ):
            res = future.result()
            results.append(res)

    # Convert results to DataFrame
    df = pd.DataFrame(results)

    # Filter out failed cases
    df = df[df["bias"].notna()]

    df.to_csv(output_csv, index=False)

    print(f"\n📁 Results saved to: {output_csv}")
    print(f"✅ Successfully generated {len(df)} bias entries, skipped {len(results) - len(df)} files.")


# ================= Main Entry =================
if __name__ == "__main__":
    process_pdb_folder(
        folder_path="ligand_binder/copys",
        output_csv="ligand_binder/copys/bias_summary_1_2_re.csv"
    )

Processing PDBs: 100%|██████████████████████████████████████████| 2000/2000 [00:31<00:00, 64.46it/s]



📁 Results saved to: ligand_binder/copys/bias_summary_1_2_re.csv
✅ Successfully generated 2000 bias entries, skipped 0 files.
